In [ ]:
!sudo apt-get update -qq
!sudo apt-get install -y libassimp-dev
!pip install -q --upgrade "hyperdrone[examples]"
!pip install -q foundation-policy==1.0.1

In [ ]:
import math
from pathlib import Path

import imageio.v2 as imageio
import numpy as np
from foundation_policy import Raptor

from hyperdrone import dynamics, render
from hyperdrone.examples.data import procthor_scene_path, x500_model_path

WIDTH, HEIGHT = 128, 128
STEPS, FPS = 180, 30
START = np.array([-3.92, -5.67, 1.0], dtype=np.float32)

scene = render.load_scene(procthor_scene_path(), fidelity="medium")
assets = render.AssetPool()
drone_asset = assets.add_assembly(render.load_assembly(x500_model_path(), fidelity="medium"))

sim = dynamics.Sim(num_drones=1, model="crazyflie", device="auto")
sim.reset(seed=0, sample_states=False)
sim.state["position"] = START[None]
sim.state["linear_velocity"] = np.array([[.25, 0, 0]], dtype=np.float32)
policy = Raptor()
policy.reset()
previous_action = np.zeros((1, sim.action_dim), dtype=np.float32)

renderer = render.Renderer(
    width=WIDTH, height=HEIGHT, num_cameras=2, output="rgb", fidelity="medium",
    num_overlays=1, max_overlay_instances=8, max_overlays_per_camera=1,
)
renderer.init(scene, assets)
renderer.attach(0, 0)  # onboard camera sees its own drone
renderer.attach(1, 0)  # external camera sees the same drone
placement = renderer.spawn(0, drone_asset, render.make_transform(position=START))
renderer.update()

# Body-frame camera above the fuselage and pitched down for self-occlusion.
pitch = .3
c, s = math.cos(pitch), math.sin(pitch)
MOUNT = np.array([[c, 0, s, .1], [0, 1, 0, 0], [-s, 0, c, .32]], dtype=np.float32)


In [ ]:
frames = []
for _ in range(STEPS):
    observation = sim.observe()
    observation[:, :3] -= START  # Raptor controls position relative to its target.
    action = policy.evaluate_step(np.concatenate([observation, previous_action], axis=1))
    sim.step(action)
    previous_action = action

    position = sim.state.numpy("position")[0]
    orientation = sim.state.numpy("orientation")[0]
    transform = render.make_transform(position=position, orientation_wxyz=orientation)
    renderer.set_transform(0, placement, transform)
    renderer.update()

    onboard = sim.camera_bases_numpy(mount=MOUNT, fov=math.radians(100), aspect=renderer.aspect)
    external = renderer.camera(
        position=position + np.array([-1.2, -1.2, .7]),
        look_at=position,
        fov=math.radians(55),
    ).reshape(1, 12)
    renderer.set_cameras(np.concatenate([onboard, external]))
    renderer.render("rgb")
    views = renderer.frame()[..., :3]
    frames.append(np.concatenate([views[0], views[1]], axis=1))

mp4_path = Path("hyperdrone_sim.mp4")
gif_path = Path("hyperdrone_sim.gif")
imageio.mimsave(mp4_path, frames, fps=FPS)
imageio.mimsave(gif_path, frames[::2], duration=2 / FPS, loop=0)
print(f"Saved {mp4_path} and {gif_path} (Raptor onboard/self-occlusion | external chase)")

In [ ]:
from IPython.display import Image, Video, display

display(Video(str(mp4_path), embed=True))
display(Image(filename=str(gif_path)))